# 📊 Data Warehouse Nilai Akademik Mahasiswa
## Program Studi Sains Data — Universitas Brawijaya — T.A. 2025/2026

**Oleh:** Nathanael Komang Bagus Prakarsa | NIM: 245091107111005

---

Notebook ini adalah **one-stop pipeline** yang mencakup:

1. 🗄️ Inisialisasi & verifikasi database (SQLite, Star Schema)
2. 📋 Query analitik pada Data Warehouse
3. 📈 Visualisasi dashboard premium (dark mode)
4. 💾 Export chart ke folder `output/charts/`

> **Catatan Privasi:** Nama dan NIM mahasiswa telah **dianonimkan**
> (contoh: *Mahasiswa 01*, *STD2500001*). File Excel asli tidak diubah.

---


## 🔧 Cell 0 — Instalasi Dependencies

In [1]:
import subprocess, sys

packages = ["pandas", "matplotlib", "seaborn", "numpy", "openpyxl", "Pillow"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Semua dependensi siap!")


Semua dependensi siap!


## ⚙️ Cell 1 — Import & Konfigurasi

In [2]:
import os, sqlite3, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from IPython.display import display, Image as IPyImage

# ─── PATH ─────────────────────────────────────────────────────────
BASE_DIR      = os.path.abspath(os.path.dirname("__file__")) if "__file__" in dir() else os.getcwd()
DB_PATH       = os.path.join(BASE_DIR, "data", "dwh_nilai_anon.db")
OUTPUT_CHARTS = os.path.join(BASE_DIR, "output", "charts")
os.makedirs(OUTPUT_CHARTS, exist_ok=True)

print(f"Working dir : {BASE_DIR}")
print(f"DB path     : {DB_PATH}")
print(f"DB ditemukan: {os.path.exists(DB_PATH)}")


Working dir : d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen
DB path     : d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\data\dwh_nilai_anon.db
DB ditemukan: True


## 🎨 Cell 2 — Design System (Dark Mode)

In [3]:
BG_COLOR      = "#0F0F13"
CARD_COLOR    = "#1A1A24"
PANEL_COLOR   = "#22222E"
TEXT_COLOR    = "#E8E8F0"
ACCENT_BLUE   = "#4F8EF7"
ACCENT_PURPLE = "#9B72F5"
ACCENT_GREEN  = "#22D3A3"
ACCENT_YELLOW = "#FBBF24"
ACCENT_RED    = "#F87171"
ACCENT_CYAN   = "#38BDF8"
ACCENT_ORANGE = "#FB923C"
GRID_COLOR    = "#2A2A38"

plt.rcParams.update({
    "figure.facecolor" : BG_COLOR,
    "axes.facecolor"   : CARD_COLOR,
    "text.color"       : TEXT_COLOR,
    "axes.labelcolor"  : TEXT_COLOR,
    "xtick.color"      : TEXT_COLOR,
    "ytick.color"      : TEXT_COLOR,
    "font.family"      : "DejaVu Sans",
    "grid.color"       : GRID_COLOR,
    "grid.linestyle"   : "--",
    "grid.linewidth"   : 0.5,
    "axes.edgecolor"   : GRID_COLOR,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})
print("Design system siap!")


Design system siap!


## 📊 Cell 3 — Query Analitik & KPI

**Star Schema** yang digunakan:

```
Dim_Mahasiswa ─────┐
Dim_MataKuliah ────┼──► Fact_Nilai
Dim_KomponenNilai ─┤
Dim_Periode ───────┘
```


In [4]:
conn = sqlite3.connect(DB_PATH)

# ── Q6: KPI ────────────────────────────────────────────────────────
kpi = pd.read_sql_query("""
    SELECT
        (SELECT COUNT(DISTINCT mahasiswa_id) FROM Dim_Mahasiswa)        AS total_mahasiswa,
        (SELECT COUNT(*) FROM Fact_Nilai)                               AS total_records,
        (SELECT COUNT(*) FROM Fact_Nilai WHERE status_nilai='lengkap') AS total_lengkap,
        (SELECT ROUND(AVG(nilai_angka),2) FROM Fact_Nilai
         WHERE status_nilai='lengkap')                                 AS avg_overall,
        (SELECT ROUND(AVG(nilai_angka),2) FROM Fact_Nilai
         WHERE status_nilai='lengkap' AND tipe_kelas='praktikum')     AS avg_praktikum,
        (SELECT ROUND(AVG(nilai_angka),2) FROM Fact_Nilai
         WHERE status_nilai='lengkap' AND tipe_kelas='teori')         AS avg_teori
""", conn)

print("=" * 50)
print("  KEY PERFORMANCE INDICATORS")
print("=" * 50)
display(kpi.T.rename(columns={0: "Nilai"}))

# ── Q1: Progress ───────────────────────────────────────────────────
df_progress = pd.read_sql_query("""
    SELECT tipe_kelas, status_nilai, COUNT(*) AS jumlah
    FROM Fact_Nilai GROUP BY tipe_kelas, status_nilai
""", conn)
print("\n=== Q1: PROGRESS KELENGKAPAN DATA ===")
display(df_progress)

# ── Q3: Rata-rata per matkul ───────────────────────────────────────
df_matkul = pd.read_sql_query("""
    SELECT mk.nama_matkul, f.tipe_kelas,
           ROUND(AVG(f.nilai_angka),2) AS rata_rata,
           COUNT(DISTINCT f.mahasiswa_id) AS jml_mhs
    FROM Fact_Nilai f
    JOIN Dim_MataKuliah mk ON f.matkul_id = mk.matkul_id
    WHERE f.status_nilai = 'lengkap'
    GROUP BY mk.matkul_id, f.tipe_kelas ORDER BY mk.nama_matkul
""", conn)
print("\n=== Q3: RATA-RATA NILAI PER MATA KULIAH ===")
display(df_matkul)

# ── Q4: Komponen Praktikum ─────────────────────────────────────────
df_komponen = pd.read_sql_query("""
    SELECT mk.nama_matkul, k.nama_komponen, k.bobot_persen,
           ROUND(AVG(f.nilai_angka),2) AS rata_rata, COUNT(f.nilai_angka) AS n
    FROM Fact_Nilai f
    JOIN Dim_MataKuliah mk ON f.matkul_id = mk.matkul_id
    JOIN Dim_KomponenNilai k ON f.komponen_id = k.komponen_id
    WHERE f.status_nilai = 'lengkap' AND f.tipe_kelas = 'praktikum'
    GROUP BY mk.matkul_id, k.komponen_id
""", conn)
print("\n=== Q4: KOMPONEN PRAKTIKUM ===")
display(df_komponen)

# ── Q5: Top 10 ─────────────────────────────────────────────────────
df_top10 = pd.read_sql_query("""
    SELECT m.nama AS nama_mahasiswa, mk.nama_matkul,
           ROUND(SUM(f.nilai_angka * (k.bobot_persen / 100.0)), 2) AS nilai_akhir_estimasi
    FROM Fact_Nilai f
    JOIN Dim_Mahasiswa m     ON f.mahasiswa_id = m.mahasiswa_id
    JOIN Dim_MataKuliah mk   ON f.matkul_id    = mk.matkul_id
    JOIN Dim_KomponenNilai k ON f.komponen_id  = k.komponen_id
    WHERE f.tipe_kelas = 'praktikum' AND f.nilai_angka IS NOT NULL
    GROUP BY m.mahasiswa_id, mk.matkul_id
    ORDER BY nilai_akhir_estimasi DESC LIMIT 10
""", conn)
print("\n=== Q5: TOP 10 NILAI AKHIR PRAKTIKUM ===")
display(df_top10)

conn.close()


  KEY PERFORMANCE INDICATORS


,Nilai
total_mahasiswa,57.00
total_records,740.00
total_lengkap,482.00
avg_overall,79.46
avg_praktikum,92.66
avg_teori,57.25



=== Q1: PROGRESS KELENGKAPAN DATA ===


,tipe_kelas,status_nilai,jumlah
0,praktikum,belum_dikoreksi,201
1,praktikum,lengkap,315
2,teori,belum_dikoreksi,1
3,teori,belum_dilaksanakan,56
4,teori,lengkap,167



=== Q3: RATA-RATA NILAI PER MATA KULIAH ===


,nama_matkul,tipe_kelas,rata_rata,jml_mhs
0,Metode Sains Data I,praktikum,84.07,29
1,Metode Statistika I,praktikum,92.38,29
2,Metode Statistika II,praktikum,96.19,28
3,Metode Statistika II,teori,57.25,56



=== Q4: KOMPONEN PRAKTIKUM ===


,nama_matkul,nama_komponen,bobot_persen,rata_rata,n
0,Metode Statistika I,Tugas 1,15,90.53,29
1,Metode Statistika I,Tugas 2,15,92.86,29
2,Metode Statistika I,Presensi,15,100.00,29
3,Metode Statistika I,Sikap,5,92.55,29
4,Metode Statistika I,UTP,20,86.04,29
5,Metode Statistika I,UAP,30,92.26,29
6,Metode Statistika II,Tugas 1,15,95.67,12
7,Metode Statistika II,Presensi,15,97.32,28
8,Metode Statistika II,Sikap,5,100.00,28
9,Metode Statistika II,UTP,20,85.17,12



=== Q5: TOP 10 NILAI AKHIR PRAKTIKUM ===


,nama_mahasiswa,nama_matkul,nilai_akhir_estimasi
0,Mahasiswa 30,Metode Statistika I,98.65
1,Mahasiswa 13,Metode Statistika I,97.40
2,Mahasiswa 06,Metode Statistika I,97.20
3,Mahasiswa 11,Metode Statistika I,97.05
4,Mahasiswa 03,Metode Statistika I,96.75
5,Mahasiswa 08,Metode Statistika I,96.55
6,Mahasiswa 02,Metode Statistika I,96.50
7,Mahasiswa 23,Metode Statistika I,96.10
8,Mahasiswa 10,Metode Statistika I,95.90
9,Mahasiswa 29,Metode Statistika I,95.80


## 📈 Cell 4 — Chart 1: Progress Kelengkapan Data

In [5]:
conn = sqlite3.connect(DB_PATH)
df1 = pd.read_sql_query(
    "SELECT tipe_kelas, status_nilai, COUNT(*) AS j FROM Fact_Nilai GROUP BY tipe_kelas, status_nilai",
    conn); conn.close()

pv = df1.pivot(index="tipe_kelas", columns="status_nilai", values="j").fillna(0)
for c in ["lengkap","belum_dikoreksi","belum_dilaksanakan"]:
    if c not in pv.columns: pv[c] = 0
pv = pv[["lengkap","belum_dikoreksi","belum_dilaksanakan"]]
pp = pv.div(pv.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=BG_COLOR)
ax.set_facecolor(CARD_COLOR)
lefts = np.zeros(len(pp)); yp = np.arange(len(pp))
colors_bar = [ACCENT_GREEN, ACCENT_YELLOW, ACCENT_RED]
labels_bar = ["Lengkap", "Belum Dikoreksi", "Belum Dilaksanakan"]

for col, clr, lbl in zip(pp.columns, colors_bar, labels_bar):
    vs = pp[col].values
    ax.barh(yp, vs, left=lefts, color=clr, height=0.5,
            edgecolor=BG_COLOR, linewidth=0.8, label=lbl)
    for idx, (v, l) in enumerate(zip(vs, lefts)):
        if v > 7:
            ax.text(l + v/2, idx, f"{v:.1f}%", ha="center", va="center",
                    color="white", fontsize=10, fontweight="bold")
    lefts += vs

ax.set_yticks(yp)
ax.set_yticklabels(["Praktikum", "Teori"], fontsize=12, fontweight="bold")
ax.set_xlim(0, 100); ax.set_xlabel("Persentase (%)", fontsize=11)
ax.set_title("Chart 1 — Progress Kelengkapan Data Nilai per Tipe Kelas",
             fontsize=13, fontweight="bold", pad=15, color=TEXT_COLOR)
ax.grid(True, axis="x", alpha=0.4)
lg = ax.legend(loc="lower right", fontsize=9, frameon=True,
               facecolor=PANEL_COLOR, edgecolor=GRID_COLOR)
[t.set_color(TEXT_COLOR) for t in lg.get_texts()]

plt.tight_layout()
p1 = os.path.join(OUTPUT_CHARTS, "chart1_progress.png")
plt.savefig(p1, dpi=200, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()
print(f"Tersimpan: {p1}")


Tersimpan: d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\output\charts\chart1_progress.png


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\4078751245.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📈 Cell 5 — Chart 2: Distribusi Nilai Teori vs Praktikum

In [6]:
conn = sqlite3.connect(DB_PATH)
df2 = pd.read_sql_query(
    "SELECT tipe_kelas, nilai_angka FROM Fact_Nilai"
    " WHERE status_nilai='lengkap' AND nilai_angka IS NOT NULL", conn)
conn.close()

fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG_COLOR)
ax.set_facecolor(CARD_COLOR)
sns.violinplot(x="tipe_kelas", y="nilai_angka", data=df2,
               palette={"praktikum": ACCENT_BLUE, "teori": ACCENT_PURPLE},
               ax=ax, inner="quartile", linewidth=1.2, cut=0.5)

means2 = df2.groupby("tipe_kelas")["nilai_angka"].mean()
stds2  = df2.groupby("tipe_kelas")["nilai_angka"].std()
for i, tp in enumerate(["praktikum", "teori"]):
    if tp in means2.index:
        ax.scatter(i, means2[tp], color=ACCENT_YELLOW, zorder=5, s=70, marker="D")
        ax.annotate(f"mean={means2[tp]:.1f}\nstd={stds2[tp]:.1f}",
                    xy=(i, means2[tp]), xytext=(i+0.3, means2[tp]+6),
                    fontsize=9, color=ACCENT_YELLOW, fontweight="bold",
                    arrowprops=dict(arrowstyle="->", color=ACCENT_YELLOW, lw=1.2),
                    bbox=dict(boxstyle="round,pad=0.3", facecolor=PANEL_COLOR,
                              edgecolor=ACCENT_YELLOW, alpha=0.9))

ax.set_title("Chart 2 — Distribusi Nilai: Teori vs Praktikum",
             fontsize=13, fontweight="bold", pad=15, color=TEXT_COLOR)
ax.set_xlabel("Tipe Kelas", fontsize=11)
ax.set_ylabel("Nilai Angka", fontsize=11)
ax.set_xticklabels(["Praktikum", "Teori"], fontsize=11, fontweight="bold")
ax.set_ylim(0, 115); ax.grid(True, axis="y", alpha=0.4)

plt.tight_layout()
p2 = os.path.join(OUTPUT_CHARTS, "chart2_distribusi.png")
plt.savefig(p2, dpi=200, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()
print(f"Tersimpan: {p2}")


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\4214882264.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x="tipe_kelas", y="nilai_angka", data=df2,
C:\Users\natha\AppData\Local\Temp\ipykernel_27652\4214882264.py:29: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["Praktikum", "Teori"], fontsize=11, fontweight="bold")


Tersimpan: d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\output\charts\chart2_distribusi.png


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\4214882264.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📈 Cell 6 — Chart 3: Rata-rata Nilai per Mata Kuliah

In [7]:
conn = sqlite3.connect(DB_PATH)
df3 = pd.read_sql_query("""
    SELECT mk.nama_matkul, f.tipe_kelas,
           ROUND(AVG(f.nilai_angka),2) AS rata_rata
    FROM Fact_Nilai f
    JOIN Dim_MataKuliah mk ON f.matkul_id = mk.matkul_id
    WHERE f.status_nilai = 'lengkap'
    GROUP BY mk.matkul_id, f.tipe_kelas
""", conn); conn.close()

df3["matkul_short"] = (df3["nama_matkul"]
    .str.replace("Metode ","M.",regex=False)
    .str.replace("Statistika","Stat.",regex=False)
    .str.replace("Sains Data","S.D.",regex=False))

fig, ax = plt.subplots(figsize=(9, 5), facecolor=BG_COLOR)
ax.set_facecolor(CARD_COLOR)
sns.barplot(x="matkul_short", y="rata_rata", hue="tipe_kelas", data=df3,
            palette={"praktikum": ACCENT_BLUE, "teori": ACCENT_PURPLE},
            ax=ax, edgecolor=BG_COLOR, width=0.6)

for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.1f}", (p.get_x()+p.get_width()/2, h+1.5),
                    ha="center", va="bottom", color=TEXT_COLOR,
                    fontsize=9.5, fontweight="bold")

ax.axhline(y=80, color=ACCENT_YELLOW, linestyle="--", linewidth=1.2, alpha=0.7)
ax.text(ax.get_xlim()[1]*0.98, 82, "KKM=80", ha="right",
        fontsize=9, color=ACCENT_YELLOW, alpha=0.9)
ax.set_ylim(0, 115)
ax.set_title("Chart 3 — Rata-rata Nilai per Mata Kuliah",
             fontsize=13, fontweight="bold", pad=15, color=TEXT_COLOR)
ax.set_xlabel("Mata Kuliah", fontsize=11); ax.set_ylabel("Rata-rata Nilai", fontsize=11)
ax.grid(True, axis="y", alpha=0.4)
lg3 = ax.legend(title="Tipe Kelas", facecolor=PANEL_COLOR, edgecolor=GRID_COLOR, fontsize=9)
lg3.get_title().set_color(TEXT_COLOR); [t.set_color(TEXT_COLOR) for t in lg3.get_texts()]

plt.tight_layout()
p3 = os.path.join(OUTPUT_CHARTS, "chart3_matkul.png")
plt.savefig(p3, dpi=200, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()
print(f"Tersimpan: {p3}")


Tersimpan: d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\output\charts\chart3_matkul.png


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\828663718.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📈 Cell 7 — Chart 4: Distribusi Komponen Praktikum

In [8]:
conn = sqlite3.connect(DB_PATH)
df4 = pd.read_sql_query("""
    SELECT mk.nama_matkul, k.nama_komponen, f.nilai_angka
    FROM Fact_Nilai f
    JOIN Dim_MataKuliah mk ON f.matkul_id = mk.matkul_id
    JOIN Dim_KomponenNilai k ON f.komponen_id = k.komponen_id
    WHERE f.status_nilai = 'lengkap'
      AND f.tipe_kelas   = 'praktikum'
      AND f.nilai_angka IS NOT NULL
""", conn); conn.close()

valid_komps = ["Tugas 1","Tugas 2","Presensi","Sikap","UTP","UAP"]
df4 = df4[df4["nama_komponen"].isin(valid_komps)]

fig, ax = plt.subplots(figsize=(11, 5.5), facecolor=BG_COLOR)
ax.set_facecolor(CARD_COLOR)
matkul_names = df4["nama_matkul"].unique()
pal = [ACCENT_BLUE, ACCENT_PURPLE, ACCENT_GREEN][:len(matkul_names)]

sns.boxplot(x="nama_komponen", y="nilai_angka", hue="nama_matkul",
            data=df4, order=valid_komps, palette=pal, ax=ax, linewidth=1.1,
            flierprops=dict(marker="o", markerfacecolor=ACCENT_RED,
                            markersize=4, markeredgecolor=BG_COLOR))

ax.axhline(y=80, color=ACCENT_YELLOW, linestyle="--", linewidth=1.2, alpha=0.7)
ax.set_ylim(40, 115)
ax.set_title("Chart 4 — Distribusi Nilai per Komponen Kelas Praktikum",
             fontsize=13, fontweight="bold", pad=15, color=TEXT_COLOR)
ax.set_xlabel("Komponen Nilai", fontsize=11); ax.set_ylabel("Nilai Mahasiswa", fontsize=11)
ax.grid(True, axis="y", alpha=0.4)
lg4 = ax.legend(title="Mata Kuliah", loc="lower right",
                facecolor=PANEL_COLOR, edgecolor=GRID_COLOR, fontsize=8.5)
lg4.get_title().set_color(TEXT_COLOR); [t.set_color(TEXT_COLOR) for t in lg4.get_texts()]

plt.tight_layout()
p4 = os.path.join(OUTPUT_CHARTS, "chart4_komponen.png")
plt.savefig(p4, dpi=200, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()
print(f"Tersimpan: {p4}")


Tersimpan: d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\output\charts\chart4_komponen.png


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\571065719.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📈 Cell 8 — Chart 5: Top 10 Mahasiswa (Data Anonim)

In [9]:
conn = sqlite3.connect(DB_PATH)
df5 = pd.read_sql_query("""
    SELECT m.nama, mk.nama_matkul,
           ROUND(SUM(f.nilai_angka*(k.bobot_persen/100.0)),2) AS nilai_akhir
    FROM Fact_Nilai f
    JOIN Dim_Mahasiswa m ON f.mahasiswa_id = m.mahasiswa_id
    JOIN Dim_MataKuliah mk ON f.matkul_id = mk.matkul_id
    JOIN Dim_KomponenNilai k ON f.komponen_id = k.komponen_id
    WHERE f.tipe_kelas = 'praktikum' AND f.nilai_angka IS NOT NULL
    GROUP BY m.mahasiswa_id, mk.matkul_id
    ORDER BY nilai_akhir DESC LIMIT 10
""", conn); conn.close()

df5s = df5.sort_values("nilai_akhir", ascending=True)
df5s["label"] = (df5s["nama"] + "\n(" +
    df5s["nama_matkul"].apply(lambda x: "".join([w[0] for w in x.split()])) + ")")

norm = df5s["nilai_akhir"] - df5s["nilai_akhir"].min()
norm = norm / (norm.max() + 0.01)
bar_colors = plt.cm.plasma(0.4 + norm.values * 0.5)

fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG_COLOR)
ax.set_facecolor(CARD_COLOR)
bars = ax.barh(range(len(df5s)), df5s["nilai_akhir"],
               color=bar_colors, edgecolor=BG_COLOR, linewidth=0.8, height=0.65)
ax.set_yticks(range(len(df5s)))
ax.set_yticklabels(df5s["label"].values, fontsize=8.5)
ax.set_xlim(85, 102)
ax.set_title("Chart 5 — Top 10 Nilai Akhir Praktikum (Data Dianonimkan)",
             fontsize=13, fontweight="bold", pad=15, color=TEXT_COLOR)
ax.set_xlabel("Nilai Akhir Estimasi", fontsize=11)
ax.grid(True, axis="x", alpha=0.4)
for bar, val in zip(bars, df5s["nilai_akhir"]):
    ax.text(val+0.1, bar.get_y()+bar.get_height()/2, f" {val:.2f}",
            va="center", ha="left", fontsize=9, color=ACCENT_YELLOW, fontweight="bold")

plt.tight_layout()
p5 = os.path.join(OUTPUT_CHARTS, "chart5_top10_anon.png")
plt.savefig(p5, dpi=200, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()
print(f"Tersimpan: {p5}")


Tersimpan: d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\output\charts\chart5_top10_anon.png


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\55523188.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 🖼️ Cell 9 — Export Dashboard Terpadu

In [10]:
# Load semua data
conn = sqlite3.connect(DB_PATH)
df_kpi = pd.read_sql_query("""
    SELECT
        (SELECT COUNT(DISTINCT mahasiswa_id) FROM Dim_Mahasiswa)        AS total_mhs,
        (SELECT COUNT(*) FROM Fact_Nilai)                               AS total_records,
        (SELECT COUNT(*) FROM Fact_Nilai WHERE status_nilai='lengkap') AS total_lengkap,
        (SELECT ROUND(AVG(nilai_angka),2) FROM Fact_Nilai
         WHERE status_nilai='lengkap')                                 AS avg_nilai,
        (SELECT ROUND(AVG(nilai_angka),2) FROM Fact_Nilai
         WHERE status_nilai='lengkap' AND tipe_kelas='praktikum')     AS avg_prak,
        (SELECT ROUND(AVG(nilai_angka),2) FROM Fact_Nilai
         WHERE status_nilai='lengkap' AND tipe_kelas='teori')         AS avg_teori
""", conn)
da = pd.read_sql_query("SELECT tipe_kelas,status_nilai,COUNT(*) j FROM Fact_Nilai GROUP BY tipe_kelas,status_nilai", conn)
db = pd.read_sql_query("SELECT tipe_kelas,nilai_angka FROM Fact_Nilai WHERE status_nilai='lengkap' AND nilai_angka IS NOT NULL", conn)
dc = pd.read_sql_query("SELECT mk.nama_matkul,f.tipe_kelas,ROUND(AVG(f.nilai_angka),2) r FROM Fact_Nilai f JOIN Dim_MataKuliah mk ON f.matkul_id=mk.matkul_id WHERE f.status_nilai='lengkap' GROUP BY mk.matkul_id,f.tipe_kelas", conn)
dd = pd.read_sql_query("SELECT mk.nama_matkul,k.nama_komponen,f.nilai_angka FROM Fact_Nilai f JOIN Dim_MataKuliah mk ON f.matkul_id=mk.matkul_id JOIN Dim_KomponenNilai k ON f.komponen_id=k.komponen_id WHERE f.status_nilai='lengkap' AND f.tipe_kelas='praktikum' AND f.nilai_angka IS NOT NULL", conn)
de = pd.read_sql_query("SELECT m.nama,mk.nama_matkul,ROUND(SUM(f.nilai_angka*(k.bobot_persen/100.0)),2) v FROM Fact_Nilai f JOIN Dim_Mahasiswa m ON f.mahasiswa_id=m.mahasiswa_id JOIN Dim_MataKuliah mk ON f.matkul_id=mk.matkul_id JOIN Dim_KomponenNilai k ON f.komponen_id=k.komponen_id WHERE f.tipe_kelas='praktikum' AND f.nilai_angka IS NOT NULL GROUP BY m.mahasiswa_id,mk.matkul_id ORDER BY v DESC LIMIT 10", conn)
conn.close()

kpi = df_kpi.iloc[0]
fig = plt.figure(figsize=(22, 16), facecolor=BG_COLOR)

# Header
hdr = fig.add_axes([0, 0.94, 1, 0.06])
hdr.set_facecolor("#1C1C2E"); hdr.axis("off")
hdr.text(0.5, 0.65, "DASHBOARD ANALITIK  |  Data Warehouse Nilai Akademik Mahasiswa",
         ha="center", va="center", fontsize=16, fontweight="bold", color="white")
hdr.text(0.5, 0.18, "Program Studi Sains Data  |  Universitas Brawijaya  |  T.A. 2025/2026  |  Data Anonim",
         ha="center", va="center", fontsize=9, color="#9999BB")

# KPI Cards
card_info = [
    ("Total Mahasiswa",   f"{int(kpi['total_mhs'])}",    ACCENT_BLUE),
    ("Total Records",     f"{int(kpi['total_records'])}", ACCENT_PURPLE),
    ("Nilai Lengkap",     f"{int(kpi['total_lengkap'])}", ACCENT_GREEN),
    ("Avg. Praktikum",    f"{kpi['avg_prak']:.2f}",      ACCENT_YELLOW),
    ("Avg. Teori",        f"{kpi['avg_teori']:.2f}",     ACCENT_CYAN),
    ("Avg. Overall",      f"{kpi['avg_nilai']:.2f}",     ACCENT_ORANGE),
]
for i, (lbl, val, clr) in enumerate(card_info):
    ca = fig.add_axes([i*(1/6)+0.003, 0.86, (1/6)-0.006, 0.075])
    ca.set_facecolor(PANEL_COLOR); ca.axis("off")
    ca.add_patch(FancyBboxPatch((0,0.85),1,0.15,transform=ca.transAxes,
                                boxstyle="square,pad=0",facecolor=clr,alpha=0.8))
    ca.text(0.5,0.52,val,ha="center",va="center",fontsize=20,fontweight="bold",color=clr,transform=ca.transAxes)
    ca.text(0.5,0.18,lbl,ha="center",va="center",fontsize=8.5,color="#AAAACC",transform=ca.transAxes)

gs = gridspec.GridSpec(2,3,left=0.04,right=0.97,top=0.845,bottom=0.04,hspace=0.38,wspace=0.32)

# Chart 1
ax1=fig.add_subplot(gs[0,0]); ax1.set_facecolor(CARD_COLOR)
pv=da.pivot(index="tipe_kelas",columns="status_nilai",values="j").fillna(0)
for c in ["lengkap","belum_dikoreksi","belum_dilaksanakan"]:
    if c not in pv.columns: pv[c]=0
pp=pv[["lengkap","belum_dikoreksi","belum_dilaksanakan"]].div(pv.sum(axis=1),axis=0)*100
lefts=np.zeros(len(pp)); yp=np.arange(len(pp))
for col,clr in zip(pp.columns,[ACCENT_GREEN,ACCENT_YELLOW,ACCENT_RED]):
    vs=pp[col].values; ax1.barh(yp,vs,left=lefts,color=clr,height=0.5,edgecolor=BG_COLOR,label=col.replace("_"," ").title())
    for idx,(v,l) in enumerate(zip(vs,lefts)):
        if v>8: ax1.text(l+v/2,idx,f"{v:.1f}%",ha="center",va="center",color="white",fontsize=9,fontweight="bold")
    lefts+=vs
ax1.set_yticks(yp); ax1.set_yticklabels(["Praktikum","Teori"],fontsize=10,fontweight="bold")
ax1.set_xlim(0,100); ax1.set_xlabel("%",fontsize=9); ax1.grid(True,axis="x",alpha=0.4)
ax1.set_title("Progress Kelengkapan Data",fontsize=11,fontweight="bold",pad=10,color=TEXT_COLOR)
l1=ax1.legend(loc="lower right",fontsize=7,frameon=True,facecolor=PANEL_COLOR,edgecolor=GRID_COLOR)
[t.set_color(TEXT_COLOR) for t in l1.get_texts()]

# Chart 2
ax2=fig.add_subplot(gs[0,1]); ax2.set_facecolor(CARD_COLOR)
sns.violinplot(x="tipe_kelas",y="nilai_angka",data=db,
               palette={"praktikum":ACCENT_BLUE,"teori":ACCENT_PURPLE},
               ax=ax2,inner="quartile",linewidth=1.2,cut=0.5)
m2=db.groupby("tipe_kelas")["nilai_angka"].mean(); s2=db.groupby("tipe_kelas")["nilai_angka"].std()
for i,tp in enumerate(["praktikum","teori"]):
    if tp in m2.index:
        ax2.scatter(i,m2[tp],color=ACCENT_YELLOW,zorder=5,s=60,marker="D")
        ax2.annotate(f"mean={m2[tp]:.1f}\nstd={s2[tp]:.1f}",xy=(i,m2[tp]),xytext=(i+0.3,m2[tp]+6),
                     fontsize=8,color=ACCENT_YELLOW,fontweight="bold",
                     arrowprops=dict(arrowstyle="->",color=ACCENT_YELLOW,lw=1),
                     bbox=dict(boxstyle="round,pad=0.3",facecolor=PANEL_COLOR,edgecolor=ACCENT_YELLOW,alpha=0.85))
ax2.set_title("Distribusi Nilai: Teori vs Praktikum",fontsize=11,fontweight="bold",pad=10,color=TEXT_COLOR)
ax2.set_xlabel("Tipe Kelas",fontsize=9); ax2.set_ylabel("Nilai",fontsize=9)
ax2.set_xticklabels(["Praktikum","Teori"],fontsize=10,fontweight="bold")
ax2.set_ylim(0,115); ax2.grid(True,axis="y",alpha=0.4)

# Chart 3
ax3=fig.add_subplot(gs[0,2]); ax3.set_facecolor(CARD_COLOR)
dc["ms"]=(dc["nama_matkul"].str.replace("Metode ","M.",regex=False)
          .str.replace("Statistika","Stat.",regex=False).str.replace("Sains Data","S.D.",regex=False))
sns.barplot(x="ms",y="r",hue="tipe_kelas",data=dc,
            palette={"praktikum":ACCENT_BLUE,"teori":ACCENT_PURPLE},ax=ax3,edgecolor=BG_COLOR,width=0.6)
for p in ax3.patches:
    h=p.get_height()
    if h>0: ax3.annotate(f"{h:.1f}",(p.get_x()+p.get_width()/2,h+1),ha="center",va="bottom",color=TEXT_COLOR,fontsize=8.5,fontweight="bold")
ax3.axhline(y=80,color=ACCENT_YELLOW,linestyle="--",linewidth=1,alpha=0.7)
ax3.set_ylim(0,115); ax3.set_title("Rata-rata Nilai per Mata Kuliah",fontsize=11,fontweight="bold",pad=10,color=TEXT_COLOR)
ax3.set_xlabel("",fontsize=9); ax3.set_ylabel("Rata-rata",fontsize=9); ax3.grid(True,axis="y",alpha=0.4)
l3=ax3.legend(title="Tipe",facecolor=PANEL_COLOR,edgecolor=GRID_COLOR,fontsize=8); l3.get_title().set_color(TEXT_COLOR)
[t.set_color(TEXT_COLOR) for t in l3.get_texts()]

# Chart 4
ax4=fig.add_subplot(gs[1,0:2]); ax4.set_facecolor(CARD_COLOR)
vk=["Tugas 1","Tugas 2","Presensi","Sikap","UTP","UAP"]
df4f=dd[dd["nama_komponen"].isin(vk)]
mn=df4f["nama_matkul"].unique()
sns.boxplot(x="nama_komponen",y="nilai_angka",hue="nama_matkul",data=df4f,order=vk,
            palette=[ACCENT_BLUE,ACCENT_PURPLE,ACCENT_GREEN][:len(mn)],ax=ax4,linewidth=1.1,
            flierprops=dict(marker="o",markerfacecolor=ACCENT_RED,markersize=4,markeredgecolor=BG_COLOR))
ax4.axhline(y=80,color=ACCENT_YELLOW,linestyle="--",linewidth=1,alpha=0.7)
ax4.set_ylim(40,115); ax4.set_title("Distribusi Nilai per Komponen Praktikum",fontsize=11,fontweight="bold",pad=10,color=TEXT_COLOR)
ax4.set_xlabel("Komponen",fontsize=9); ax4.set_ylabel("Nilai",fontsize=9); ax4.grid(True,axis="y",alpha=0.4)
l4=ax4.legend(title="Mata Kuliah",loc="lower right",facecolor=PANEL_COLOR,edgecolor=GRID_COLOR,fontsize=8)
l4.get_title().set_color(TEXT_COLOR); [t.set_color(TEXT_COLOR) for t in l4.get_texts()]

# Chart 5
ax5=fig.add_subplot(gs[1,2]); ax5.set_facecolor(CARD_COLOR)
df5s=de.sort_values("v",ascending=True)
df5s["lbl"]=df5s["nama"]+"\n("+df5s["nama_matkul"].apply(lambda x:"".join([w[0] for w in x.split()]))+")"
nrm=df5s["v"]-df5s["v"].min(); nrm=nrm/(nrm.max()+0.01)
bc=plt.cm.plasma(0.4+nrm.values*0.5)
bars5=ax5.barh(range(len(df5s)),df5s["v"],color=bc,edgecolor=BG_COLOR,linewidth=0.8,height=0.65)
ax5.set_yticks(range(len(df5s))); ax5.set_yticklabels(df5s["lbl"].values,fontsize=7.5)
ax5.set_xlim(85,102); ax5.set_title("Top 10 Nilai Akhir Praktikum",fontsize=11,fontweight="bold",pad=10,color=TEXT_COLOR)
ax5.set_xlabel("Nilai Akhir",fontsize=9); ax5.grid(True,axis="x",alpha=0.4)
for bar,val in zip(bars5,df5s["v"]):
    ax5.text(val+0.1,bar.get_y()+bar.get_height()/2,f" {val:.2f}",va="center",ha="left",fontsize=8,color=ACCENT_YELLOW,fontweight="bold")

# Footer
ft=fig.add_axes([0,0,1,0.018]); ft.set_facecolor("#12121C"); ft.axis("off")
ft.text(0.5,0.5,"Nathanael Komang Bagus Prakarsa  |  NIM 245091107111005  |  Sains Data - Universitas Brawijaya  |  2026",
        ha="center",va="center",fontsize=8,color="#666688")

dash_path = os.path.join(OUTPUT_CHARTS, "dashboard_combined_anon.png")
plt.savefig(dash_path, dpi=180, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()
print(f"Dashboard terpadu tersimpan: {dash_path}")


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\284918595.py:72: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x="tipe_kelas",y="nilai_angka",data=db,
C:\Users\natha\AppData\Local\Temp\ipykernel_27652\284918595.py:85: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax2.set_xticklabels(["Praktikum","Teori"],fontsize=10,fontweight="bold")


Dashboard terpadu tersimpan: d:\Nael\Mata Kuliah Sains Data\.vscode\Kuliah\Semester 4\DWH Dosen\output\charts\dashboard_combined_anon.png


C:\Users\natha\AppData\Local\Temp\ipykernel_27652\284918595.py:137: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ✅ Cell 10 — Ringkasan Hasil

| Tahap | Deskripsi | Status |
|-------|-----------|--------|
| **Skema DWH** | Star Schema: 4 dimensi + 1 tabel fakta | ✅ |
| **ETL** | Excel → Transformasi → SQLite | ✅ |
| **Anonimisasi** | 57 mahasiswa disamarkan | ✅ |
| **Query Analitik** | 6 query SQL tersimpan di `sql/dwh_queries.sql` | ✅ |
| **Visualisasi** | 5 chart premium + 1 dashboard terpadu | ✅ |

### Referensi:
- Kimball, R., & Ross, M. (2013). *The Data Warehouse Toolkit*. Wiley.
- McKinney, W. (2010). Data Structures for Statistical Computing in Python.
- SQLite Documentation: https://www.sqlite.org/docs.html
